In [ ]:
# -*- coding: utf-8 -*-
"""
STANDALONE inference + Open3D visualization script.
Completely independent — no import from, or dependency on, benchmark.ipynb or any
other notebook/file. Everything this needs (loaders, features, model classes) is
defined right here.

Usage:
    visualize_inference_dir(
        infer_dir   = "data/test",
        model_path  = "checkpoints/best_seg_ClassicalML_RandomForest.joblib",
        model_kind  = "sklearn",          # "sklearn"  or  "torch"
        model_name  = "RandomForest",     # only needed when model_kind == "torch"
        max_files   = 5,
    )

Requires: numpy, open3d, joblib, laspy (for .las/.laz), torch (+ torch_cluster /
torch_scatter only if you load a PointTransformer/GNN checkpoint).
"""
import os
import glob
import numpy as np
import open3d as o3d
import joblib

# ============================================================== 1) POINT-CLOUD LOADER
LABEL_KEYS = ("label", "labels", "classification", "class",
              "scalar_label", "scalar_classification", "seg", "pred")


def load_pointcloud(path):
    """Returns (points[N,3] float64). Supports .las/.laz/.ply/.pcd/.xyz/.pts/.txt"""
    ext = os.path.splitext(path)[1].lower()

    if ext in (".las", ".laz"):
        import laspy
        las = laspy.read(path)
        pts = np.column_stack((np.asarray(las.x), np.asarray(las.y),
                               np.asarray(las.z))).astype(np.float64)
        return pts

    if ext == ".ply":
        pcd = o3d.io.read_point_cloud(path)
        return np.asarray(pcd.points, dtype=np.float64)

    if ext == ".pcd":
        pcd = o3d.io.read_point_cloud(path)
        return np.asarray(pcd.points, dtype=np.float64)

    # .xyz / .pts / .txt
    for kw in (dict(), dict(delimiter=","), dict(skiprows=1),
               dict(delimiter=",", skiprows=1)):
        try:
            arr = np.loadtxt(path, **kw)
            break
        except ValueError:
            arr = None
    if arr is None:
        arr = np.genfromtxt(path, delimiter=",", skip_header=1)
    arr = np.asarray(arr, dtype=np.float64)
    return arr[:, :3]


def list_pointcloud_files(folder):
    exts = (".las", ".laz", ".ply", ".pcd", ".xyz", ".pts", ".txt")
    out = []
    for e in exts:
        out += glob.glob(os.path.join(folder, "*" + e))
    return sorted(out)


# ============================================================== 2) HANDCRAFTED FEATURES (for sklearn models)
def point_features(pts, k=24):
    """Per-point eigen/geometry features from a kNN neighbourhood.
    pts: (N,3) array (any coordinate scale). Returns (N,8) float32 array:
    [z_rel, |normal_z|, linearity, planarity, sphericity, curvature,
     log-density, local height-range]
    Must match the feature order used when the sklearn model was trained
    (n_points, in that order) — same 8 features as in the benchmark notebook.
    """
    n = len(pts)
    z = pts[:, 2]
    z_rel = (z - z.min()) / max(z.max() - z.min(), 1e-9)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts.astype(np.float64))
    tree = o3d.geometry.KDTreeFlann(pcd)
    neigh = np.empty((n, k), np.int64)
    for i in range(n):
        _, idx, _ = tree.search_knn_vector_3d(pts[i], k)
        idx = list(idx)[:k]
        if len(idx) < k:                       # pad if cloud smaller than k
            idx = idx + [idx[-1]] * (k - len(idx))
        neigh[i] = idx

    nb = pts[neigh]                            # (N,k,3)
    mu = nb.mean(1, keepdims=True)
    cov = np.einsum("nki,nkj->nij", nb - mu, nb - mu) / k
    w, V = np.linalg.eigh(cov)                 # ascending eigenvalues
    l3, l2, l1 = w[:, 0], w[:, 1], w[:, 2]
    s = np.clip(l1, 1e-12, None)
    linearity = (l1 - l2) / s
    planarity = (l2 - l3) / s
    sphericity = l3 / s
    curvature = l3 / np.clip(l1 + l2 + l3, 1e-12, None)
    nz_abs = np.abs(V[:, 2, 0])                 # normal = eigenvector of smallest eigval
    r_k = np.linalg.norm(nb[:, -1, :] - pts, axis=1)
    density = k / np.clip((4.0 / 3.0) * np.pi * r_k ** 3, 1e-9, None)
    height_range = nb[:, :, 2].max(1) - nb[:, :, 2].min(1)

    X = np.column_stack([z_rel, nz_abs, linearity, planarity, sphericity,
                         curvature, np.log1p(density), height_range]).astype(np.float32)
    return np.nan_to_num(X)


# ============================================================== 3) SKLEARN MODEL INFERENCE
def predict_sklearn(model_path, pts, batch=100_000):
    """model_path: joblib file with a dict {'model':.., 'scaler':.., ...}
    (the format the benchmark notebook saves classical-ML checkpoints in)."""
    blob = joblib.load(model_path)
    clf, scaler = blob["model"], blob["scaler"]
    n = len(pts)
    out = np.empty(n, np.int64)
    for start in range(0, n, batch):
        idx = np.arange(start, min(start + batch, n))
        feats = point_features(pts, k=24)[idx] if n <= batch else \
            point_features(pts[idx], k=24)          # small-batch features
        out[idx] = clf.predict(scaler.transform(feats))
    return out


# ============================================================== 4) TORCH MODEL INFERENCE (optional path)
def predict_torch(model_path, model_name, pts, num_points=4096, batch=8, device=None):
    """model_path: .pth file with {'model_state':..., 'num_classes':...}.
    model_name must be one of the architectures defined below.
    Requires torch (+ torch_cluster/torch_scatter for PointTransformer/GNN models)."""
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    num_classes = ckpt.get("num_classes", 2)

    # ---- minimal geometry helpers (self-contained, no external deps) ----
    def square_distance(a, b):
        return ((a[:, :, None, :] - b[:, None, :, :]) ** 2).sum(-1)

    def index_points(p, idx):
        B = p.shape[0]
        view = [B] + [1] * (idx.dim() - 1)
        bidx = torch.arange(B, device=p.device).view(view).expand_as(idx)
        return p[bidx, idx]

    def farthest_point_sample(xyz, npoint):
        B, N, _ = xyz.shape
        idx = torch.zeros(B, npoint, dtype=torch.long, device=xyz.device)
        dist = torch.full((B, N), 1e10, device=xyz.device)
        far = torch.randint(0, N, (B,), device=xyz.device)
        b = torch.arange(B, device=xyz.device)
        for i in range(npoint):
            idx[:, i] = far
            d = ((xyz - xyz[b, far].unsqueeze(1)) ** 2).sum(-1)
            dist = torch.minimum(dist, d)
            far = dist.argmax(-1)
        return idx

    def query_ball_point(radius, nsample, xyz, new_xyz):
        B, N, _ = xyz.shape
        nsample = min(nsample, N)
        S = new_xyz.shape[1]
        group = torch.arange(N, device=xyz.device).view(1, 1, N).repeat(B, S, 1)
        group[square_distance(new_xyz, xyz) > radius ** 2] = N
        group = group.sort(-1)[0][:, :, :nsample]
        first = group[:, :, :1].expand(-1, -1, nsample)
        mask = group == N
        group[mask] = first[mask]
        return group

    class SetAbstraction(nn.Module):
        def __init__(self, npoint, radius, nsample, in_ch, mlp):
            super().__init__()
            self.npoint, self.radius, self.nsample = npoint, radius, nsample
            layers, last = [], in_ch + 3
            for out in mlp:
                layers += [nn.Conv2d(last, out, 1), nn.BatchNorm2d(out), nn.ReLU()]
                last = out
            self.mlp = nn.Sequential(*layers)

        def forward(self, xyz, feats):
            xyz_t = xyz.permute(0, 2, 1)
            fps = farthest_point_sample(xyz_t, self.npoint)
            new_xyz = index_points(xyz_t, fps)
            idx = query_ball_point(self.radius, self.nsample, xyz_t, new_xyz)
            grouped = index_points(xyz_t, idx) - new_xyz.unsqueeze(2)
            if feats is not None:
                grouped = torch.cat([grouped, index_points(feats.permute(0, 2, 1), idx)], -1)
            new_feats = self.mlp(grouped.permute(0, 3, 2, 1)).max(2)[0]
            return new_xyz.permute(0, 2, 1), new_feats

    class FeaturePropagation(nn.Module):
        def __init__(self, in_ch, mlp):
            super().__init__()
            layers, last = [], in_ch
            for out in mlp:
                layers += [nn.Conv1d(last, out, 1), nn.BatchNorm1d(out), nn.ReLU()]
                last = out
            self.mlp = nn.Sequential(*layers)

        def forward(self, xyz1, xyz2, f1, f2):
            x1, x2 = xyz1.permute(0, 2, 1), xyz2.permute(0, 2, 1)
            d = square_distance(x1, x2)
            d, idx = d.sort(-1)
            d, idx = d[:, :, :3].clamp(min=1e-10), idx[:, :, :3]
            w = (1.0 / d); w = w / w.sum(-1, keepdim=True)
            interp = (index_points(f2.permute(0, 2, 1), idx) * w.unsqueeze(-1)).sum(2)
            out = interp.permute(0, 2, 1)
            if f1 is not None:
                out = torch.cat([f1, out], 1)
            return self.mlp(out)

    IN_CH = 3   # this standalone script uses raw xyz only (no height/normal channels)

    class PointNetSeg(nn.Module):
        def __init__(self, num_classes):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Conv1d(IN_CH, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
                nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
                nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU())
            self.mid = nn.Sequential(nn.Conv1d(IN_CH, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
                                     nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
            self.head = nn.Sequential(
                nn.Conv1d(1024 + 128, 256, 1), nn.BatchNorm1d(256), nn.ReLU(),
                nn.Conv1d(256, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
                nn.Dropout(0.4), nn.Conv1d(128, num_classes, 1))

        def forward(self, x):
            g = self.enc(x).max(-1)[0]
            local = self.mid(x)
            gexp = g.unsqueeze(-1).expand(-1, -1, x.shape[-1])
            return self.head(torch.cat([gexp, local], 1))

    class PointNet2Seg(nn.Module):
        def __init__(self, num_classes):
            super().__init__()
            self.sa1 = SetAbstraction(1024, 0.1, 32, IN_CH, [32, 32, 64])
            self.sa2 = SetAbstraction(256, 0.2, 32, 64 + 3, [64, 64, 128])
            self.sa3 = SetAbstraction(64, 0.4, 32, 128 + 3, [128, 128, 256])
            self.sa4 = SetAbstraction(16, 0.8, 32, 256 + 3, [256, 256, 512])
            self.fp4 = FeaturePropagation(512 + 256, [256, 256])
            self.fp3 = FeaturePropagation(256 + 128, [256, 256])
            self.fp2 = FeaturePropagation(256 + 64, [256, 128])
            self.fp1 = FeaturePropagation(128, [128, 128, 128])
            self.head = nn.Sequential(nn.Conv1d(128, 128, 1), nn.BatchNorm1d(128),
                                      nn.ReLU(), nn.Dropout(0.4),
                                      nn.Conv1d(128, num_classes, 1))

        def forward(self, x):
            l1x, l1f = self.sa1(x, x)
            l2x, l2f = self.sa2(l1x, l1f)
            l3x, l3f = self.sa3(l2x, l2f)
            l4x, l4f = self.sa4(l3x, l3f)
            l3f = self.fp4(l3x, l4x, l3f, l4f)
            l2f = self.fp3(l2x, l3x, l2f, l3f)
            l1f = self.fp2(l1x, l2x, l1f, l2f)
            return self.head(self.fp1(x, l1x, None, l1f))

    MODEL_REGISTRY = {"PointNet": PointNetSeg, "PointNet++": PointNet2Seg}
    if model_name not in MODEL_REGISTRY:
        raise ValueError(
            f"'{model_name}' architecture is not bundled in this standalone script. "
            f"Available here: {list(MODEL_REGISTRY)}. "
            f"(PointTransformer/GNN/PointNeXt/DGCNN/PointMLP classes can be added the "
            f"same way if you need them — ask and I'll drop them in.)")

    model = MODEL_REGISTRY[model_name](num_classes).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    # ---- chunked full-cloud inference ----
    center = pts.mean(0, keepdims=True)
    scale = max(np.linalg.norm(pts - center, axis=1).max(), 1e-9)
    pts_n = ((pts - center) / scale).astype(np.float32)
    n = len(pts_n)
    perm = np.random.RandomState(0).permutation(n)
    pad = (num_points - n % num_points) % num_points
    if pad:
        perm = np.concatenate([perm, perm[:pad]])
    chunks = perm.reshape(-1, num_points)
    out = np.zeros(n, np.int64)
    with torch.no_grad():
        for i in range(0, len(chunks), batch):
            cid = chunks[i:i + batch]
            x = torch.from_numpy(pts_n[cid].transpose(0, 2, 1)).to(device)
            pred = model(x).argmax(1).cpu().numpy()
            out[cid.ravel()] = pred.ravel()
    return out


# ============================================================== 5) VISUALIZATION
def visualize_inference_dir(infer_dir, model_path, model_kind="sklearn",
                            model_name=None, target_class=1, max_files=None,
                            num_points=4096, window_size=(1280, 800)):
    """
    infer_dir   : folder of point-cloud files to run inference on
    model_path  : path to the saved model checkpoint
    model_kind  : "sklearn" (joblib) or "torch" (.pth)
    model_name  : required if model_kind == "torch" (e.g. "PointNet", "PointNet++")
    target_class: label id to paint GREEN (everything else is dark gray)
    """
    files = list_pointcloud_files(infer_dir)
    if max_files:
        files = files[:max_files]
    if not files:
        print(f"No point-cloud files found in: {infer_dir}")
        return
    print(f"Found {len(files)} file(s) in '{infer_dir}' — model: {model_path}")

    for fp in files:
        pts = load_pointcloud(fp)

        if model_kind == "sklearn":
            pred = predict_sklearn(model_path, pts)
        elif model_kind == "torch":
            if model_name is None:
                raise ValueError("model_name is required when model_kind='torch'")
            pred = predict_torch(model_path, model_name, pts, num_points=num_points)
        else:
            raise ValueError("model_kind must be 'sklearn' or 'torch'")

        colors = np.zeros((len(pts), 3))
        colors[pred == target_class] = [0.0, 0.80, 0.0]     # GREEN = target class
        colors[pred != target_class] = [0.88, 0, 0]   # dark gray = others

        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(pts.astype(np.float64)-np.mean(pts.astype(np.float64),axis=0))
        pcd.colors = o3d.utility.Vector3dVector(colors)

        n_target = int((pred == target_class).sum())
        title = (f"{os.path.basename(fp)} | green(class {target_class})="
                 f"{n_target} | gray(others)={len(pts) - n_target}")
        o3d.visualization.draw_geometries([pcd], window_name=title,
                                          width=window_size[0], height=window_size[1])


# ============================================================== USAGE EXAMPLE
if __name__ == "__main__":
    # --- classical ML checkpoint (RandomForest / XGBoost / etc.) ---
    visualize_inference_dir(
        infer_dir="data/test",
        model_path="checkpoints/best_seg_ClassicalML_RandomForest.joblib",
        model_kind="sklearn",
        target_class=1,
        max_files=10,
    )

    # --- torch checkpoint (PointNet / PointNet++ only, in this standalone file) ---
    # visualize_inference_dir(
    #     infer_dir="data/test",
    #     model_path="checkpoints/best_seg_DeepLearning_PointNet.pth",
    #     model_kind="torch",
    #     model_name="PointNet",
    #     target_class=1,
    #     max_files=5,
    # )

Found 10 file(s) in 'data/test' — model: checkpoints/best_seg_ClassicalML_RandomForest.joblib


### Randomforest
- 50
- 53
- 54
- 55
- 